# Kv2.1 MolProbity / clashscore analysis

All six WT/L403A/F412L × vanilla/masked ensembles are compared. The conformational extension
uses the exact structures and existing chain-resolved coordinates from
`Kv21_L403A_conformational_validation.ipynb`. Established tetramer/interface/alignment QC is retained.

## Reproducible setup
Run from the repository root or channel directory. Dependencies: NumPy, pandas, SciPy, statsmodels, Matplotlib, seaborn, and Jupyter. MolProbity tables remain in the local downloaded directory. No upload or move is performed. Shared functions live in `shared/molprobity_analysis.py`, as in the existing analyses.

In [1]:
from pathlib import Path
import sys, os, json, platform, importlib.metadata
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'shared/data_access.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from shared import molprobity_analysis as mp
CHANNEL = 'kv21'
OUT = mp.output_dir(CHANNEL)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 24)
mp.apply_kv21_style()


## Data-access boundary
Set `VGIC_MOLPROBITY_ROOT` or `DATA_ROOT` to the downloaded directory. Future Hugging Face migration only needs to replace the loader with explicit paths through the existing `shared.data_access.resolve`; downstream analysis accepts the same DataFrame. Existing structural-QC and conformational inputs already use that resolver, including local/cache/offline behavior.

In [2]:
DATA_ROOT = os.environ.get('VGIC_MOLPROBITY_ROOT')
raw, inventory, total_csv_files = mp.load_molprobity(CHANNEL, DATA_ROOT)
inventory.to_csv(OUT/'tables/input_inventory.csv', index=False)
print(f'{total_csv_files} CSVs in downloaded directory; {len(inventory)} for this channel; {len(raw):,} rows')
display(inventory)


18 CSVs in downloaded directory; 6 for this channel; 35,989 rows


,csv_file,rows,sha256
0,Kv21_f412l_masked_common99_50pct_clashscores.csv,6000,2e1cc802c1e4027223cdc9a538aaee1c63dd29b5973246...
1,Kv21_f412l_vanilla_clashscores.csv,6000,6805e145186fa2566a9151b437e1d006f4c7d4931988de...
2,Kv21_l403a_masked_common99_50pct_clashscores.csv,5995,1a3157a729dc7c84d41ee96c858de7d18b800a516c66f1...
3,Kv21_l403a_vanilla_clashscores.csv,5994,021390d94f175d45838b4e9dede98a20b71808c77e39b2...
4,Kv21_wt_masked_common99_50pct_clashscores.csv,6000,37940638140c47aeacbe7576f1ff160e838b6211788696...
5,Kv21_wt_vanilla_clashscores.csv,6000,ad9c53ab87b8228adc075727b07ce483e87cb7f56a5b3a...


## Audit before plotting
The expected grid uses each ensemble’s recorded seed labels, AF models 1–5, r0–r10, and final. Missing whole trajectories/seeds are additionally checked against the existing model manifest. Raw input hashes are saved above. NaN recycle is expected only for final/base models; missing rank is allowed and reported.

In [3]:
data, audit = mp.audit_data(raw, CHANNEL, OUT)
display(audit)
display(pd.read_csv(OUT/'tables/missing_snapshots.csv'))
display(pd.read_csv(OUT/'tables/manifest_discrepancies.csv'))
display(pd.read_csv(OUT/'tables/row_anomalies.csv'))
cohorts = mp.select_cohorts(data, CHANNEL, OUT)
summary = pd.concat([mp.describe(frame, name) for name, frame in cohorts.items()], ignore_index=True)
summary.to_csv(OUT/'tables/descriptive_statistics.csv', index=False)
display(summary[(summary.cohort=='all_recorded') & summary.metric.eq('clashscore')].round(3))
display(pd.read_csv(OUT/'tables/selection_audit.csv'))


/Users/ahernandezgonzalez/miniforge3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Audit: 35,989 rows, 0 failed, 0 invalid metric rows; 11 absent expected snapshots; 0 explicitly excluded rows.


,ensemble,rows,ok,ok_fraction,eligible,failed,missing_clashscore,nonfinite_clashscore,duplicate_names,duplicate_keys,missing_grid_snapshots,seeds,seed_min,seed_max,missing_rank,final_r10_pairs,final_r10_identical_metrics,manifest_rows,manifest_missing,manifest_extra
0,F412L | masked,6000,6000,1.0,6000,0,0,0,0,0,0,100,33,132,0,500,500,6000,0,0
1,F412L | vanilla,6000,6000,1.0,6000,0,0,0,0,0,0,100,78,177,0,500,500,6000,0,0
2,L403A | masked,5995,5995,1.0,5995,0,0,0,0,0,5,100,33,132,0,498,498,5995,0,0
3,L403A | vanilla,5994,5994,1.0,5994,0,0,0,0,0,6,100,55,154,5994,499,499,5994,0,0
4,WT | masked,6000,6000,1.0,6000,0,0,0,0,0,0,100,33,132,0,500,500,6000,0,0
5,WT | vanilla,6000,6000,1.0,6000,0,0,0,0,0,0,100,48,147,0,500,500,6000,0,0


,ensemble,seed,af_model,snapshot
0,L403A | masked,85,5,r10
1,L403A | masked,85,5,r7
2,L403A | masked,85,5,r8
3,L403A | masked,85,5,r9
4,L403A | masked,124,5,r10
5,L403A | vanilla,154,5,final
6,L403A | vanilla,154,5,r10
7,L403A | vanilla,154,5,r6
8,L403A | vanilla,154,5,r7
9,L403A | vanilla,154,5,r8


,ensemble,model_name,issue


,run_name,input_folder,model_name,rank,af_model,seed,recycle,snapshot,clashscore,n_bad_clashes,max_overlap,status,...,source_path,genotype,protocol,ensemble,csv_file,analysis_eligible,malformed_numeric,invalid_identity,duplicate_name,duplicate_key,invalid_metric,failed_status_or_exit


,cohort,ensemble,metric,N,n_seeds,median,Q1,Q3,IQR,mean,SD,minimum,maximum
0,all_recorded,F412L | masked,clashscore,6000,100,52.545,45.358,74.712,29.355,61.220,23.157,28.83,295.23
3,all_recorded,F412L | vanilla,clashscore,6000,100,45.810,36.228,55.920,19.693,48.977,16.940,22.60,218.63
6,all_recorded,L403A | masked,clashscore,5995,100,48.500,41.240,61.780,20.540,54.319,19.593,27.02,259.78
9,all_recorded,L403A | vanilla,clashscore,5994,100,45.785,38.585,57.598,19.013,50.238,17.231,22.65,203.36
12,all_recorded,WT | masked,clashscore,6000,100,50.210,44.520,65.335,20.815,58.565,21.874,25.79,251.79
15,all_recorded,WT | vanilla,clashscore,6000,100,44.640,38.400,56.415,18.015,49.229,16.020,24.45,143.01


,ensemble,selection,source,requested,matched,excluded_failed,excluded_by_structural_qc
0,F412L | masked,3 Å + G377/pore–VSD interface QC,kv21/dataDistances/26-11-20_Kv2.1_f412l_masked...,2543,2543,0,3457
1,F412L | vanilla,3 Å + G377/pore–VSD interface QC,kv21/dataDistances/26-02-11_Kv2.1_f412l_vanill...,4521,4521,0,1479
2,L403A | masked,3 Å + G377/pore–VSD interface QC,kv21/dataDistances/26-11-19_Kv2.1_l403a_masked...,3610,3610,0,2385
3,L403A | vanilla,3 Å + G377/pore–VSD interface QC,kv21/dataDistances/26-02-11_Kv2.1_l403a_vanill...,4403,4403,0,1591
4,WT | masked,3 Å + G377/pore–VSD interface QC,kv21/dataDistances/26-11-20_Kv2.1_wt_masked_co...,1529,1529,0,4471
5,WT | vanilla,3 Å + G377/pore–VSD interface QC,kv21/dataDistances/26-02-11_Kv2.1_wt_vanillaAF...,4591,4591,0,1409


## Statistical design and interpretation

The complete downloaded table is audited before filtering. `status != ok`, nonzero exit codes,
nonfinite/negative diagnostics and malformed identities are reported explicitly. Ambiguous identities
or mismatched manifests stop execution. Empty anomaly CSVs still retain their headers.

Every ensemble has up to 100 seeds × 5 AF models × 11 recycle snapshots plus a final/base file.
The final/base file has identical recorded MolProbity metrics to r10 for every available trajectory;
that is **metric equality, not a coordinate checksum**. To avoid double weighting those observations,
`all_recycles` excludes final copies. `final_only` is a separate sensitivity analysis.

The primary scientific comparison uses the repository's existing QC subset, independently of clashscore:
3 Å convergence for Cav1.2/Nav1.5 and the established structural/interface/alignment-QC distance
selection for Kv2.1. All raw distributions and excluded counts remain visible. QC-selected results
describe survivors, not all generated models. No new clashscore cutoff is used.

For each metric, average snapshots within seed/AF-model trajectory, then average AF models within seed.
This gives each retained seed equal weight and avoids treating recycles or the five AF models as
independent replicates. Point estimates are seed-balanced mean differences B−A; 95% percentile
bootstrap intervals use 3,000 seed draws. When seed-label sets match completely, nominal paired-seed
t tests and the existing `paired_common_seed_bootstrap` are used. Otherwise Welch tests compare all
retained seed summaries. Both independent-seed and common-label paired sensitivities are exported.
Shared numeric labels do **not** establish shared random-number streams; no seeds are renumbered,
and rank is never a pairing key. Common-label sensitivity may change the population when ranges differ.

BH correction covers all planned pairwise comparisons × all three metrics within each channel/cohort;
primary, independent, and common-label p-values form separate families. Supplemental cohort families
are exploratory. The seed rank-biserial effect is P(B>A)−P(B<A) across seed summaries, with ties zero;
it describes marginal ordering even for nominally paired tests. Model medians/IQRs are descriptive
snapshot summaries and are a different estimand from the seed-balanced mean contrast.

AF-model/snapshot-adjusted regressions provide a sensitivity check with standard errors clustered by
recorded seed. Rank is an output ordering variable, is absent for Kv2.1 L403A vanilla, and is inspected
descriptively rather than controlled as if it were a randomized covariate. These uncertainty estimates
describe computational sampling, not biological replication or a validated clinical effect.

Clashscore is a normalized global clash statistic ([Phenix glossary](https://www.phenix-online.org/version_docs/1.8.2-1296/glossary.htm)).
`n_bad_clashes` is an absolute count and `max_overlap` is a single extreme; they are complementary
diagnostics, not interchangeable biological measures. The local export does not record Phenix version,
hydrogen options, the atom denominator, or atom-level clash identities. Absolute validation grades,
cross-channel biological comparisons, and residue-specific attribution require additional provenance.


In [4]:
contrasts = mp.contrast_statistics(cohorts, CHANNEL, OUT)
primary = contrasts[(contrasts.cohort=='repository_QC') & contrasts.metric.eq('clashscore')]
display(primary[['A','B','N_A','N_B','n_seeds_A','n_seeds_B','common_seeds',
                 'common_model_snapshots','only_A_snapshots','only_B_snapshots',
                 'paired_seed_correlation','primary_test']])
display(primary[['A','B','median_difference','seed_mean_difference','CI_low','CI_high',
                 'seed_rank_biserial','q_BH','independent_q_BH','common_seed_difference',
                 'common_seed_CI_low','common_seed_CI_high','common_seed_q_BH']].round(4))


,A,B,N_A,N_B,n_seeds_A,n_seeds_B,common_seeds,common_model_snapshots,only_A_snapshots,only_B_snapshots,paired_seed_correlation,primary_test
27,WT | vanilla,WT | masked,4591,1529,100,100,85,1255,3336,274,-0.047087,Welch seed t
30,L403A | vanilla,L403A | masked,4403,3610,100,100,78,2548,1855,1062,-0.032224,Welch seed t
33,F412L | vanilla,F412L | masked,4521,2543,100,100,55,1323,3198,1220,-0.038751,Welch seed t
36,WT | vanilla,L403A | vanilla,4591,4403,100,100,93,3973,618,430,0.072098,Welch seed t
39,WT | vanilla,F412L | vanilla,4591,4521,100,100,70,3011,1580,1510,0.050513,Welch seed t
42,L403A | vanilla,F412L | vanilla,4403,4521,100,100,77,3170,1233,1351,0.136270,Welch seed t
45,WT | masked,L403A | masked,1529,3610,100,100,100,1331,198,2279,-0.094647,paired nominal seed t
48,WT | masked,F412L | masked,1529,2543,100,100,100,947,582,1596,-0.131555,paired nominal seed t
51,L403A | masked,F412L | masked,3610,2543,100,100,100,2237,1373,306,0.063224,paired nominal seed t


,A,B,median_difference,seed_mean_difference,CI_low,CI_high,seed_rank_biserial,q_BH,independent_q_BH,common_seed_difference,common_seed_CI_low,common_seed_CI_high,common_seed_q_BH
27,WT | vanilla,WT | masked,5.13,4.6325,3.5624,5.8072,0.5742,0.0000,0.0000,4.3076,3.0642,5.5488,0.0000
30,L403A | vanilla,L403A | masked,0.81,0.6281,-0.1873,1.4396,0.0870,0.2201,0.2165,0.6293,-0.3765,1.6279,0.2872
33,F412L | vanilla,F412L | masked,2.52,2.9874,2.1773,3.7839,0.5192,0.0000,0.0000,3.0611,1.9559,4.1881,0.0000
36,WT | vanilla,L403A | vanilla,0.92,0.3295,-0.6476,1.3206,0.0562,0.5341,0.5341,0.3748,-0.5960,1.3814,0.5099
39,WT | vanilla,F412L | vanilla,1.25,-0.3354,-1.2268,0.5183,-0.0378,0.5214,0.5214,-0.3770,-1.4725,0.6552,0.5099
42,L403A | vanilla,F412L | vanilla,0.33,-0.6648,-1.5239,0.2030,-0.1156,0.2186,0.2165,-0.8165,-1.7179,0.1397,0.1699
45,WT | masked,L403A | masked,-3.40,-3.6750,-4.7454,-2.6072,-0.5546,0.0000,0.0000,-3.6750,-4.7454,-2.6072,0.0000
48,WT | masked,F412L | masked,-1.36,-1.9805,-3.1176,-0.8692,-0.2935,0.0026,0.0011,-1.9805,-3.1176,-0.8692,0.0026
51,L403A | masked,F412L | masked,2.04,1.6945,0.9474,2.4275,0.3354,0.0001,0.0001,1.6945,0.9474,2.4275,0.0001


## Clashscore distributions and effect sizes
Violins retain every eligible individual snapshot with median and quartiles; no outliers are clipped. The paired raw/QC panels expose selection effects. The contrast figure shows seed-balanced differences and 95% bootstrap intervals; q-values are reported without significance stars. Colors, typography, thin outlines, light grids, and PDF/300-dpi PNG exports follow `shared.plotting` and existing conformational figures.

In [5]:
mp.plot_distributions(cohorts, CHANNEL, OUT)
plt.show()
mp.plot_effects(contrasts, CHANNEL, OUT)
plt.show()
display(Markdown(mp.interpretation(summary, contrasts, CHANNEL)))


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_98989/2804364640.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_98989/2804364640.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Results on the repository QC subset

Model medians describe retained snapshots. Differences and confidence intervals below use equal AF-model weights within seed and seed-level inference.
- **WT | masked versus WT | vanilla:** model medians 48.21 versus 43.08 (Δ +5.13); seed-balanced mean Δ +4.63, 95% CI [3.56, 5.81]; seed rank-biserial effect +0.57; BH q=2.64e-12.
- **L403A | masked versus L403A | vanilla:** model medians 44.81 versus 44.00 (Δ +0.81); seed-balanced mean Δ +0.63, 95% CI [-0.19, 1.44]; seed rank-biserial effect +0.09; BH q=0.22.
- **F412L | masked versus F412L | vanilla:** model medians 46.85 versus 44.33 (Δ +2.52); seed-balanced mean Δ +2.99, 95% CI [2.18, 3.78]; seed rank-biserial effect +0.52; BH q=1.09e-10.
- **L403A | vanilla versus WT | vanilla:** model medians 44.00 versus 43.08 (Δ +0.92); seed-balanced mean Δ +0.33, 95% CI [-0.65, 1.32]; seed rank-biserial effect +0.06; BH q=0.534.
- **F412L | vanilla versus WT | vanilla:** model medians 44.33 versus 43.08 (Δ +1.25); seed-balanced mean Δ -0.34, 95% CI [-1.23, 0.52]; seed rank-biserial effect -0.04; BH q=0.521.
- **F412L | vanilla versus L403A | vanilla:** model medians 44.33 versus 44.00 (Δ +0.33); seed-balanced mean Δ -0.66, 95% CI [-1.52, 0.20]; seed rank-biserial effect -0.12; BH q=0.219.
- **L403A | masked versus WT | masked:** model medians 44.81 versus 48.21 (Δ -3.40); seed-balanced mean Δ -3.67, 95% CI [-4.75, -2.61]; seed rank-biserial effect -0.55; BH q=4.22e-09.
- **F412L | masked versus WT | masked:** model medians 46.85 versus 48.21 (Δ -1.36); seed-balanced mean Δ -1.98, 95% CI [-3.12, -0.87]; seed rank-biserial effect -0.29; BH q=0.00262.
- **F412L | masked versus L403A | masked:** model medians 46.85 versus 44.81 (Δ +2.04); seed-balanced mean Δ +1.69, 95% CI [0.95, 2.43]; seed rank-biserial effect +0.34; BH q=9.58e-05.

## Complementary steric diagnostics
These panels use the repository QC subset. Counts depend on structure size and atom handling; maximum overlap reports one extreme contact. Full numerical results and corrected comparisons for both metrics are retained alongside clashscore.

In [6]:
mp.plot_diagnostics(cohorts, CHANNEL, OUT)
plt.show()
display(summary[(summary.cohort=='repository_QC') & summary.metric.ne('clashscore')].round(3))
display(contrasts[(contrasts.cohort=='repository_QC') & contrasts.metric.ne('clashscore')][
    ['A','B','metric','median_difference','seed_mean_difference','CI_low','CI_high','seed_rank_biserial','q_BH']].round(4))


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_98989/1691443703.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,cohort,ensemble,metric,N,n_seeds,median,Q1,Q3,IQR,mean,SD,minimum,maximum
55,repository_QC,F412L | masked,n_bad_clashes,2543,100,1808.000,1647.000,2000.500,353.500,1856.286,348.254,1112.000,4659.000
56,repository_QC,F412L | masked,max_overlap,2543,100,1.629,1.600,1.654,0.054,1.624,0.044,1.471,1.765
58,repository_QC,F412L | vanilla,n_bad_clashes,4521,100,1710.000,1354.000,1974.000,620.000,1750.444,488.635,873.000,4180.000
59,repository_QC,F412L | vanilla,max_overlap,4521,100,1.629,1.597,1.655,0.058,1.622,0.049,1.426,1.787
61,repository_QC,L403A | masked,n_bad_clashes,3610,100,1728.000,1510.250,2034.750,524.500,1806.032,414.057,1042.000,3980.000
62,repository_QC,L403A | masked,max_overlap,3610,100,1.628,1.597,1.653,0.056,1.623,0.044,1.444,1.766
64,repository_QC,L403A | vanilla,n_bad_clashes,4403,100,1695.000,1438.500,1975.500,537.000,1772.939,470.269,874.000,5172.000
65,repository_QC,L403A | vanilla,max_overlap,4403,100,1.627,1.599,1.653,0.054,1.624,0.045,1.441,1.780
67,repository_QC,WT | masked,n_bad_clashes,1529,100,1860.000,1710.000,2057.000,347.000,1925.012,365.716,995.000,4547.000
68,repository_QC,WT | masked,max_overlap,1529,100,1.634,1.606,1.660,0.054,1.630,0.042,1.444,1.742


,A,B,metric,median_difference,seed_mean_difference,CI_low,CI_high,seed_rank_biserial,q_BH
28,WT | vanilla,WT | masked,n_bad_clashes,198.000,179.7638,138.8026,224.7931,0.5794,0.0000
29,WT | vanilla,WT | masked,max_overlap,0.006,0.0057,0.0023,0.0093,0.2776,0.0040
31,L403A | vanilla,L403A | masked,n_bad_clashes,33.000,25.4678,-5.8116,56.4621,0.0918,0.2101
32,L403A | vanilla,L403A | masked,max_overlap,0.001,-0.0009,-0.0036,0.0016,-0.0600,0.5279
34,F412L | vanilla,F412L | masked,n_bad_clashes,98.000,116.5117,85.4654,147.0733,0.5244,0.0000
35,F412L | vanilla,F412L | masked,max_overlap,0.000,0.0024,-0.0006,0.0055,0.0892,0.2101
37,WT | vanilla,L403A | vanilla,n_bad_clashes,33.000,11.1166,-26.2760,49.1046,0.0504,0.5656
38,WT | vanilla,L403A | vanilla,max_overlap,-0.001,-0.0016,-0.0040,0.0006,-0.1040,0.2427
40,WT | vanilla,F412L | vanilla,n_bad_clashes,48.000,-12.9517,-47.0940,19.7362,-0.0390,0.5214
41,WT | vanilla,F412L | vanilla,max_overlap,0.001,-0.0028,-0.0052,-0.0004,-0.1862,0.0477


## Recycle, AF-model, seed, and rank sensitivity
The full snapshot trace includes final separately and all early-recycle tails. The final-only analysis does not substitute a last available snapshot for a missing final file. The adjusted regression controls AF-model and snapshot composition, with uncertainty clustered by seed; its observation-weighted estimand differs from the seed-balanced primary result.

In [7]:
mp.stratification(cohorts, CHANNEL, OUT)
plt.show()
adjusted = mp.adjusted_contrasts(cohorts, CHANNEL, OUT)
display(adjusted.round(4))
af_sensitivity = mp.af_model_sensitivity(cohorts, CHANNEL, OUT)
display(af_sensitivity.pivot(index=['A','B','mode'], columns='AF_model', values='seed_mean_difference').round(3))
display(contrasts[contrasts.metric.eq('clashscore')][
    ['cohort','A','B','seed_mean_difference','CI_low','CI_high','q_BH']].round(3))


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_98989/2647556288.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,A,B,adjusted_difference,CI_low,CI_high,p,q_BH
0,WT | vanilla,WT | masked,3.9936,2.9466,5.0407,0.0000,0.0000
1,L403A | vanilla,L403A | masked,0.4637,-0.2734,1.2008,0.2176,0.2798
2,F412L | vanilla,F412L | masked,2.4636,1.7031,3.2242,0.0000,0.0000
3,WT | vanilla,L403A | vanilla,0.6080,-0.3329,1.5488,0.2053,0.2798
4,WT | vanilla,F412L | vanilla,-0.3770,-1.2572,0.5032,0.4012,0.4514
5,L403A | vanilla,F412L | vanilla,-0.9584,-1.8054,-0.1114,0.0266,0.0598
6,WT | masked,L403A | masked,-0.7905,-1.7851,0.2041,0.1193,0.2147
7,WT | masked,F412L | masked,-0.1976,-1.1654,0.7702,0.6891,0.6891
8,L403A | masked,F412L | masked,1.7670,1.0946,2.4393,0.0000,0.0000


AF_model                                                 1      2      3  \
A               B               mode                                       
F412L | vanilla F412L | masked  leave_AF_model_out   1.510  4.169  1.998   
                                only_AF_model        3.954 -1.244  7.134   
L403A | masked  F412L | masked  leave_AF_model_out   2.841  1.766  1.460   
                                only_AF_model       -4.116  1.652  2.733   
L403A | vanilla F412L | vanilla leave_AF_model_out  -1.808 -0.653 -0.569   
                                only_AF_model        3.944 -0.473 -0.990   
                L403A | masked  leave_AF_model_out  -3.138  1.749 -0.031   
                                only_AF_model       12.014 -3.368  3.410   
WT | masked     F412L | masked  leave_AF_model_out  -2.105 -2.964 -1.125   
                                only_AF_model        0.726  0.610 -4.542   
                L403A | masked  leave_AF_model_out  -4.945 -4.731 -2.586   
                                only_AF_model        4.842 -1.041 -7.275   
WT | vanilla    F412L | vanilla leave_AF_model_out  -1.602 -0.695  0.164   
                                only_AF_model        4.751  1.166 -2.247   
                L403A | vanilla leave_AF_model_out   0.205 -0.041  0.733   
                                only_AF_model        0.807  1.639 -1.257   
                WT | masked     leave_AF_model_out   2.013  6.439  3.288   
                                only_AF_model        7.979 -0.688  9.429   

AF_model                                                 4      5  
A               B               mode                               
F412L | vanilla F412L | masked  leave_AF_model_out   2.435  4.579  
                                only_AF_model       -0.789 -3.611  
L403A | masked  F412L | masked  leave_AF_model_out   1.721  0.439  
                                only_AF_model       -4.632  5.319  
L403A | vanilla F412L | vanilla leave_AF_model_out   0.742 -1.036  
                                only_AF_model       -6.560  0.894  
                L403A | masked  leave_AF_model_out   1.456  3.104  
                                only_AF_model       -2.718 -8.036  
WT | masked     F412L | masked  leave_AF_model_out  -1.608 -1.700  
                                only_AF_model      -10.493  0.368  
                L403A | masked  leave_AF_model_out  -3.329 -2.139  
                                only_AF_model       -5.861 -4.951  
WT | vanilla    F412L | vanilla leave_AF_model_out   0.725 -0.269  
                                only_AF_model       -4.612 -0.398  
                L403A | vanilla leave_AF_model_out  -0.017  0.768  
                                only_AF_model        1.948 -1.292  
                WT | masked     leave_AF_model_out   4.768  6.011  
                                only_AF_model        5.092 -4.378

,cohort,A,B,seed_mean_difference,CI_low,CI_high,q_BH
0,all_recycles,WT | vanilla,WT | masked,9.673,8.592,10.810,0.000
3,all_recycles,L403A | vanilla,L403A | masked,3.980,2.936,5.039,0.000
6,all_recycles,F412L | vanilla,F412L | masked,12.398,11.340,13.373,0.000
9,all_recycles,WT | vanilla,L403A | vanilla,1.129,0.171,2.113,0.028
12,all_recycles,WT | vanilla,F412L | vanilla,-0.172,-1.067,0.722,0.709
15,all_recycles,L403A | vanilla,F412L | vanilla,-1.302,-2.163,-0.399,0.007
18,all_recycles,WT | masked,L403A | masked,-4.565,-5.743,-3.400,0.000
21,all_recycles,WT | masked,F412L | masked,2.552,1.294,3.813,0.000
24,all_recycles,L403A | masked,F412L | masked,7.117,5.939,8.283,0.000
27,repository_QC,WT | vanilla,WT | masked,4.633,3.562,5.807,0.000


## S6 conformational association: join first, test second

The existing chain-resolved table contains all six ensembles. Join on **ensemble + exact PDB basename**,
require one record per canonical chain, and cross-check seed, AF model, and snapshot. No rank-based
matching or fuzzy filename normalization is permitted. The audit covers all recorded models; the
primary association uses the exact final structural-QC subset.

Two complementary existing-coordinate summaries are examined:

- Median canonical-chain F412-position Cα displacement from 8SD3 (Å), an **unsigned** magnitude.
  For F412L, the existing column refers to the homologous L412 Cα anchor.
- A directional projection of the four canonical-chain **whole-S6 rotations** onto the experimental
  8SD3→8SDA rotation vector: `dot(model − 8SD3, 8SDA − 8SD3) / norm(8SDA − 8SD3)^2`.
  Angular differences wrap to [−180°,180°). Zero is the WT rotation reference, one the mutant-reference
  projection. This is a derived summary of the existing coordinate, not a state classifier, a complete
  3-D conformation match, or a claim that projection one reproduces all four chains. Canonical subunit
  mapping is inherited from the validated analysis, not inferred from MolProbity.

Spearman correlation uses seed summaries (median per trajectory, equal AF-model mean within seed),
with 1,000 seed bootstrap draws. Descriptive model-level rho is also exported. Regressions adjust for
AF model/snapshot and cluster uncertainty by seed; slope × coordinate IQR expresses a local magnitude.
Direct interaction tests compare slopes between conditions rather than comparing significance labels.
The 12 association tests per method/cohort and the 18 interaction tests form separate BH families.
Raw-recycle, final-only, and r10 within the same structural-QC population are sensitivity analyses. Correlation cannot establish that
steric strain causes or limits S6 movement; early-recycle quality and survivor selection can influence it.


In [8]:
joined, associations, interactions = mp.kv21_conformation(cohorts, OUT)
display(pd.read_csv(OUT/'tables/conformation_join_audit.csv'))
display(pd.read_csv(OUT/'tables/S6_rotation_reference.csv'))
display(associations[associations.cohort.eq('repository_QC')].round(4))
display(interactions.round(4))
mp.plot_conformation(joined, associations, OUT)
plt.show()
mp.plot_conformation_adjusted(joined, associations, OUT)
plt.show()
display(associations[associations.cohort.eq('r10_repository_QC')].round(4))


,ensemble,_merge,N
0,F412L | masked,both,6000
1,F412L | vanilla,both,6000
2,L403A | masked,both,5995
3,L403A | vanilla,both,5994
4,WT | masked,both,6000
5,WT | vanilla,both,6000


,canonical_subunit,8SD3,8SDA,experimental_delta
0,A,0.0,10.883062,10.883062
1,B,0.0,-38.513955,-38.513955
2,C,0.0,6.703495,6.703495
3,D,0.0,-42.517921,-42.517921


,cohort,ensemble,metric,N,missing_metric,seeds,model_spearman_rho,seed_spearman_rho,rho_CI_low,rho_CI_high,spearman_p,adjusted_slope,slope_CI_low,slope_CI_high,slope_p,metric_Q1,metric_Q3,fitted_clashscore_change_per_IQR,slope_q_BH,spearman_q_BH
0,repository_QC,F412L | masked,S6_shift_projection,2543,0,100,-0.0795,-0.0569,-0.2711,0.1647,0.5742,1.5803,0.4667,2.6940,0.0054,-0.0035,0.1131,0.1843,0.0164,0.6418
1,repository_QC,F412L | masked,F412_displacement_A,2543,0,100,0.0719,-0.0727,-0.2998,0.1523,0.4724,0.2923,0.0429,0.5418,0.0216,0.9031,1.5767,0.1969,0.0433,0.6299
2,repository_QC,F412L | vanilla,S6_shift_projection,4521,0,100,0.3373,0.1118,-0.0876,0.3162,0.2683,-0.1040,-1.4105,1.2024,0.8760,0.0695,0.1022,-0.0034,0.8760,0.4599
3,repository_QC,F412L | vanilla,F412_displacement_A,4521,0,100,0.4021,-0.1332,-0.3343,0.0959,0.1865,0.4469,-0.0274,0.9211,0.0648,0.9994,1.5860,0.2621,0.1077,0.3729
4,repository_QC,L403A | masked,S6_shift_projection,3610,0,100,0.0033,0.0015,-0.1899,0.1994,0.9886,9.5818,3.8555,15.3081,0.0010,-0.1200,-0.0042,1.1097,0.0125,0.9886
5,repository_QC,L403A | masked,F412_displacement_A,3610,0,100,0.2792,0.3477,0.1426,0.5251,0.0004,1.7606,0.5889,2.9323,0.0032,0.4983,0.9062,0.7181,0.0164,0.0024
6,repository_QC,L403A | vanilla,S6_shift_projection,4403,0,100,0.3309,0.0816,-0.1229,0.2613,0.4197,9.7650,1.9084,17.6216,0.0148,0.0402,0.0613,0.2068,0.0356,0.6296
7,repository_QC,L403A | vanilla,F412_displacement_A,4403,0,100,0.6274,0.1881,-0.0310,0.3774,0.0609,1.6529,0.4866,2.8191,0.0055,0.5776,0.7537,0.2911,0.0164,0.1828
8,repository_QC,WT | masked,S6_shift_projection,1529,0,100,-0.1679,0.0548,-0.1327,0.2485,0.5883,-0.9762,-2.0388,0.0865,0.0718,0.0043,0.4938,-0.4779,0.1077,0.6418
9,repository_QC,WT | masked,F412_displacement_A,1529,0,100,-0.0850,0.1560,-0.0299,0.3234,0.1212,-0.1830,-0.6337,0.2677,0.4261,0.4606,4.6814,-0.7725,0.5681,0.2909


,metric,A,B,slope_difference,CI_low,CI_high,p,q_BH
0,S6_shift_projection,WT | vanilla,WT | masked,1.9484,-7.0306,10.9274,0.6706,0.8047
1,S6_shift_projection,L403A | vanilla,L403A | masked,-0.1832,-10.0640,9.6977,0.9710,0.9710
2,S6_shift_projection,F412L | vanilla,F412L | masked,1.6844,-0.0722,3.4409,0.0602,0.1204
3,S6_shift_projection,WT | vanilla,L403A | vanilla,12.6895,0.5387,24.8404,0.0407,0.1036
4,S6_shift_projection,WT | vanilla,F412L | vanilla,2.8205,-6.2443,11.8854,0.5420,0.7504
5,S6_shift_projection,L403A | vanilla,F412L | vanilla,-9.8690,-17.8238,-1.9142,0.0150,0.0513
6,S6_shift_projection,WT | masked,L403A | masked,10.5579,4.6450,16.4709,0.0005,0.0084
7,S6_shift_projection,WT | masked,F412L | masked,2.5565,0.9342,4.1788,0.0020,0.0148
8,S6_shift_projection,L403A | masked,F412L | masked,-8.0014,-13.8932,-2.1097,0.0078,0.0350
9,F412_displacement_A,WT | vanilla,WT | masked,-0.9333,-3.3326,1.4660,0.4458,0.7295


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_98989/3501492492.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_98989/3501492492.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,cohort,ensemble,metric,N,missing_metric,seeds,model_spearman_rho,seed_spearman_rho,rho_CI_low,rho_CI_high,spearman_p,adjusted_slope,slope_CI_low,slope_CI_high,slope_p,metric_Q1,metric_Q3,fitted_clashscore_change_per_IQR,slope_q_BH,spearman_q_BH
12,r10_repository_QC,F412L | masked,S6_shift_projection,320,0,100,-0.1024,-0.0169,-0.2402,0.1899,0.8671,1.7072,0.4573,2.9572,0.0074,-0.0045,0.0989,0.1765,0.0297,0.8671
13,r10_repository_QC,F412L | masked,F412_displacement_A,320,0,100,0.2474,0.1203,-0.0905,0.3176,0.2330,0.7691,0.4730,1.0652,0.0000,0.9228,1.7541,0.6394,0.0000,0.4530
14,r10_repository_QC,F412L | vanilla,S6_shift_projection,487,0,100,0.1211,-0.0492,-0.2587,0.1646,0.6267,0.4544,-1.1131,2.0220,0.5699,0.0605,0.0944,0.0154,0.6839,0.6944
15,r10_repository_QC,F412L | vanilla,F412_displacement_A,487,0,100,0.3765,-0.2771,-0.4611,-0.0594,0.0052,-0.3533,-0.8892,0.1826,0.1963,0.9764,1.5925,-0.2176,0.3366,0.0315
16,r10_repository_QC,L403A | masked,S6_shift_projection,405,0,100,-0.0063,-0.0478,-0.2487,0.1416,0.6366,4.1623,-1.5476,9.8721,0.1531,-0.1388,-0.0051,0.5565,0.3062,0.6944
17,r10_repository_QC,L403A | masked,F412_displacement_A,405,0,100,0.0810,0.1834,-0.0363,0.3778,0.0678,1.5731,0.3746,2.7716,0.0101,0.4321,0.8425,0.6456,0.0303,0.2035
18,r10_repository_QC,L403A | vanilla,S6_shift_projection,483,0,100,0.1929,0.0679,-0.1318,0.2595,0.5022,6.4054,2.8190,9.9917,0.0005,0.0372,0.0593,0.1415,0.0028,0.6696
19,r10_repository_QC,L403A | vanilla,F412_displacement_A,483,0,100,0.6039,0.2276,0.0223,0.4102,0.0227,0.9830,-0.3301,2.2960,0.1423,0.5555,0.6974,0.1394,0.3062,0.0910
20,r10_repository_QC,WT | masked,S6_shift_projection,219,0,100,-0.2073,0.0923,-0.0948,0.2633,0.3610,0.1225,-1.8477,2.0927,0.9030,-0.0068,0.1067,0.0139,0.9030,0.5414
21,r10_repository_QC,WT | masked,F412_displacement_A,219,0,100,-0.1028,0.1657,-0.0402,0.3475,0.0995,0.3299,-0.7385,1.3983,0.5450,0.4612,2.8333,0.7825,0.6839,0.2388


## Focused interpretation
The following text is generated from the executed result tables so it stays synchronized with reruns.

In [9]:
display(Markdown(mp.focused_interpretation(summary, contrasts, CHANNEL, OUT)))

**Global quality and S6.** Masking changes the seed-balanced mean clashscore by WT: +4.633 (95% CI +3.562 to +5.807); L403A: +0.628 (95% CI -0.187 to +1.440); F412L: +2.987 (95% CI +2.177 to +3.784).

The original conformational table joins 35,989 available models exactly, including seed/AF-model/snapshot checks; 21,197 enter established structural QC. These are successful identity matches, not proof of coordinate checksums across cluster runs.

Directional S6 projection associations depend on aggregation and selection. For each ensemble below, compare the seed-level rank association with the AF-model/recycle-adjusted model-level slope; they estimate different relationships:

- F412L | masked: seed ρ=-0.06 (95% CI -0.27 to +0.16); adjusted slope +1.58 clashscore per projection unit, CI [0.47, 2.69], equivalent to +0.18 over that ensemble’s coordinate IQR.

- F412L | vanilla: seed ρ=+0.11 (95% CI -0.09 to +0.32); adjusted slope -0.10 clashscore per projection unit, CI [-1.41, 1.20], equivalent to -0.00 over that ensemble’s coordinate IQR.

- L403A | masked: seed ρ=+0.00 (95% CI -0.19 to +0.20); adjusted slope +9.58 clashscore per projection unit, CI [3.86, 15.31], equivalent to +1.11 over that ensemble’s coordinate IQR.

- L403A | vanilla: seed ρ=+0.08 (95% CI -0.12 to +0.26); adjusted slope +9.76 clashscore per projection unit, CI [1.91, 17.62], equivalent to +0.21 over that ensemble’s coordinate IQR.

- WT | masked: seed ρ=+0.05 (95% CI -0.13 to +0.25); adjusted slope -0.98 clashscore per projection unit, CI [-2.04, 0.09], equivalent to -0.48 over that ensemble’s coordinate IQR.

- WT | vanilla: seed ρ=+0.20 (95% CI +0.02 to +0.39); adjusted slope -2.92 clashscore per projection unit, CI [-11.84, 5.99], equivalent to -0.06 over that ensemble’s coordinate IQR.

**r10 within the same QC population:**

- F412L | masked: seed ρ=-0.02 (95% CI -0.24 to +0.19); adjusted slope +1.71, CI [0.46, 2.96].

- F412L | vanilla: seed ρ=-0.05 (95% CI -0.26 to +0.16); adjusted slope +0.45, CI [-1.11, 2.02].

- L403A | masked: seed ρ=-0.05 (95% CI -0.25 to +0.14); adjusted slope +4.16, CI [-1.55, 9.87].

- L403A | vanilla: seed ρ=+0.07 (95% CI -0.13 to +0.26); adjusted slope +6.41, CI [2.82, 9.99].

- WT | masked: seed ρ=+0.09 (95% CI -0.09 to +0.26); adjusted slope +0.12, CI [-1.85, 2.09].

- WT | vanilla: seed ρ=+0.11 (95% CI -0.10 to +0.32); adjusted slope -2.47, CI [-9.10, 4.15].

Unsigned F412-position displacement has positive adjusted slopes in the pooled QC analysis, but this is not universal across the r10 sensitivities and does not establish motion toward 8SDA. A universal steric ceiling on the experimentally directed S6 shift is not established: marginal and conditional associations differ, early-recycle behavior matters, and the analysis observes sampled survivors rather than an energetic barrier. The S6 association is exploratory and potentially figure-worthy only with the selection/adjustment sensitivity shown. Atom-level S6/interface clash localization followed by controlled relaxation of matched models would be more informative about a steric mechanism; correlation alone is not causation.

**AF-model dependence.** The following ranges are effect magnitudes, not new hypothesis tests:

- WT masked−vanilla: individual AF-model mean differences range from -4.38 to +9.43; leaving out one AF model at a time gives +2.01 to +6.44.

- L403A masked−vanilla: individual AF-model mean differences range from -8.04 to +12.01; leaving out one AF model at a time gives -3.14 to +3.10.

- F412L masked−vanilla: individual AF-model mean differences range from -3.61 to +7.13; leaving out one AF model at a time gives +1.51 to +4.58.

Mixed signs across AF models preclude a uniform per-model masking effect. The stratified tables and final-r10 sensitivities should accompany claims based on pooled means.

## Reproducibility record
All generated outputs are confined to this channel’s `dataExtra/molprobity/{tables,figures}`. Existing figures and analyses are not overwritten. Input hashes, missing-model lists, full descriptive/contrast tables, pairing diagnostics and sensitivity results accompany the figures.

In [10]:
record = {'channel': CHANNEL, 'total_downloaded_csvs': total_csv_files,
          'channel_csvs': len(inventory), 'rows': len(raw), 'python': platform.python_version(),
          'bootstrap_replicates': mp.BOOTSTRAPS, 'random_seed': mp.RANDOM_SEED,
          'packages': {p: importlib.metadata.version(p) for p in ['numpy','pandas','scipy','statsmodels','matplotlib','seaborn']}}
(OUT/'tables/run_metadata.json').write_text(json.dumps(record, indent=2)+chr(10))
print('Completed:', OUT.relative_to(ROOT))


Completed: kv21/dataExtra/molprobity
